# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. We walk through loading the dataset defined by a Croissant schema, reviewing its structure, and performing basic data analysis with proper usage of Croissant entity `@id` fields.

### Dataset Source
The dataset source is defined via a Croissant schema URL.

In [ ]:
# Install the mlcroissant library if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset (Croissant schema)
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as an object (not a dictionary)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, their `@id`s, and the list of available fields and columns in each. All references are made by their `@id` fields as per Croissant best practices.

In [ ]:
# List the available record sets and their @id fields
print("Available record sets in the dataset:")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}")

# As an example, for each record set, list its column @id and field @id
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Columns:")
    for col in rs.columns:
        print(f"   - @id: {col.id}, name: {col.name}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"   - @id: {field.id}, name: {field.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the actual record set and field `@id`s from the overview above.

In [ ]:
# Collect all the record set @id values in a list
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded dataframe with shape: {df.shape}")
        print(f"  Columns: {df.columns.tolist()}")
    else:
        print("  No records found.")

# Use the first non-empty record set for demonstration
main_record_set_id = next((rid for rid in record_set_ids if rid in dataframes), None)
if main_record_set_id:
    print(f"\nUsing RecordSet @id as main example: {main_record_set_id}\n")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
As an example, we select a numeric field using its `@id`, filter records above a threshold, normalize that column, and group or aggregate using another field.

Replace the field `@id` below with any appropriate field you identified in the Data Overview for your use case.

In [ ]:
# Pick the main dataframe
df = dataframes[main_record_set_id]

# Let's automatically pick a numeric column by checking dtypes
numeric_cols = df.select_dtypes(include='number').columns.tolist()
if len(numeric_cols) == 0:
    print("No numeric columns found in the dataframe.")
else:
    numeric_field_id = numeric_cols[0]  # Take the first numeric field
    print(f"Using numeric field for EDA: {numeric_field_id}")

    threshold = df[numeric_field_id].quantile(0.95)  # Use 95th percentile as an example threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' above 95th percentile ({threshold:.2f}): {len(filtered_df)} records.")

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try to group by a categorical column
    cat_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if len(cat_fields) > 0:
        group_field_id = cat_fields[0]  # Use the first available categorical field
        print(f"\nGrouping and aggregating by field: {group_field_id}")
        grouped_df = df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(grouped_df.head())
    else:
        print("No categorical fields available for grouping.")

## 5. Visualization
Let's plot the distribution of the numeric field and visualize the grouped aggregation (if grouping is possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution
if len(numeric_cols) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Visualize grouped mean if grouping was possible
    if len(cat_fields) > 0:
        top_categories = grouped_df.sort_values(by=numeric_field_id, ascending=False).head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(x=top_categories.index, y=numeric_field_id, data=top_categories.reset_index())
        plt.title(f'Top 10 {group_field_id} mean {numeric_field_id}')
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook provided a guided workflow for loading and interacting with a Croissant-structured dataset using `mlcroissant`, referencing all entities by their `@id`. Explore further analyses using other fields and record sets as necessary.